# Panorama da Despesa com Pessoal dos Estados Brasileiros
## Relatório de Gestão Fiscal — 2015 a 2025

Este notebook analisa quanto da Receita Corrente Líquida (RCL) está comprometido com a Despesa Total com Pessoal (DTP) dos Poderes Executivos estaduais.

**Questão central:** quais estados preservam maior espaço fiscal diante dos limites da Lei de Responsabilidade Fiscal e quais apresentam pressão, recorrência ou deterioração?

> **Escopo:** 27 UFs, exercícios de 2015 a 2025 e último quadrimestre disponível. A análise considera somente o Poder Executivo (`co_poder=E`), pois os limites legais são definidos por poder/órgão.

**Fonte:** API oficial do SICONFI/STN, RGF — Anexo 01. Percentuais e limites declarados no demonstrativo têm prioridade sobre valores recalculados.

### Como ler esta análise

1. Cobertura e qualidade dos dados;
2. situação fiscal no último exercício;
3. ranking anual e score histórico;
4. melhora, deterioração, recorrência e volatilidade;
5. gráficos e síntese para decisão.

DTP/RCL e margens são expressos em **pontos percentuais**. Menor DTP/RCL e maiores margens indicam, em geral, situação mais confortável.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, HTML, Image

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from rgf.analysis import build_indicators, annual_ranking
from rgf.charts import generate_charts
from rgf.export import executive_summary, export_excel

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
display(HTML('<style>.jp-Notebook{max-width:1250px;margin:auto}.dataframe{font-size:12px}h1,h2,h3{color:#19324D}blockquote{border-left:4px solid #167D5A;padding-left:12px}</style>'))
base = pd.read_csv(ROOT / 'dados/rgf_estados_2015_2025_tratado.csv')
indicadores = build_indicators(base)
ranking = annual_ranking(base)
ultimo_ano = int(base.dropna(subset=['DTP_RCL']).ano.max())

## 1. Cobertura e consistência

A grade esperada possui 297 observações: 27 UFs multiplicadas por 11 exercícios. Dados ausentes permanecem nulos; não são substituídos por zero.

O leiaute do RGF ganhou detalhamento a partir de 2018. Para manter a comparabilidade, o tratamento combina código e descrição normalizada da conta. A origem de cada indicador pode ser auditada em `dados/auditoria_mapeamento.csv`.

In [ ]:
cobertura = pd.DataFrame({
    'Indicador': ['Observações esperadas', 'DTP/RCL disponível', 'UFs no último ano', 'Exercícios'],
    'Resultado': [297, int(base.DTP_RCL.notna().sum()), int(base.query('ano == @ultimo_ano').UF.nunique()), int(base.ano.nunique())]
})
display(cobertura.style.hide(axis='index').set_caption('Resumo da cobertura'))
ausencias = pd.read_csv(ROOT / 'dados/ausencias_por_ano.csv')
display(ausencias.style.hide(axis='index').set_caption('Ausências por exercício'))

## 2. Situação fiscal no último exercício

A classificação compara o DTP/RCL aos limites declarados no próprio RGF. **Confortável** indica distância relevante do alerta; **atenção** sinaliza aproximação; **alerta** representa superação do limite de alerta; as duas últimas categorias indicam ultrapassagem dos limites prudencial ou máximo.

> A classificação é gerencial. O enquadramento legal definitivo depende do demonstrativo homologado e da avaliação dos órgãos competentes.

In [ ]:
atual = base.query('ano == @ultimo_ano').copy()
ordem = ['confortável','atenção','alerta','acima do limite prudencial','acima do limite máximo']
situacao = atual.situacao_fiscal.value_counts().reindex(ordem, fill_value=0).rename_axis('Situação').reset_index(name='UFs')
display(Markdown(f'**Exercício de referência: {ultimo_ano}.**'))
display(situacao.style.hide(axis='index').set_caption('Distribuição da situação fiscal'))
cols = ['UF','nome_estado','DTP_RCL','Limite_Alerta','Limite_Prudencial','Limite_Maximo','Margem_Limite_Prudencial','situacao_fiscal']
pressao = atual.query("situacao_fiscal != 'confortável'")[cols].sort_values('DTP_RCL', ascending=False)
display(pressao.style.format({c:'{:.2f}' for c in cols[2:7]}, na_rep='—').hide(axis='index').set_caption('Estados que exigem atenção'))
regional = atual.groupby('regiao', as_index=False).agg(DTP_RCL_medio=('DTP_RCL','mean'), margem_prudencial_media=('Margem_Limite_Prudencial','mean')).sort_values('DTP_RCL_medio')
display(regional.style.format({'DTP_RCL_medio':'{:.2f}','margem_prudencial_media':'{:.2f}'}).hide(axis='index').set_caption('Comparação regional'))

## 3. Ranking e score fiscal

O ranking anual ordena as UFs do menor para o maior DTP/RCL. Como uma fotografia isolada não captura trajetória e recorrência, o ranking histórico usa um score de 0 a 100.

$$Score = 100 \times (0{,}30N + 0{,}20P + 0{,}15M + 0{,}15R + 0{,}10T + 0{,}10V)$$

- **N — nível (30%):** menor média histórica do DTP/RCL;
- **P — margem prudencial (20%):** maior distância média do prudencial;
- **M — margem máxima (15%):** maior distância média do máximo;
- **R — recorrência (15%):** menos anos acima do alerta;
- **T — tendência (10%):** trajetória recente mais favorável;
- **V — volatilidade (10%):** maior estabilidade anual.

Cada componente é transformado em percentil entre as 27 UFs. Os grupos representam os terços superior, central e inferior do score.

In [ ]:
ra = atual[['ranking_ano','UF','nome_estado','DTP_RCL','Limite_Prudencial','Margem_Limite_Prudencial','situacao_fiscal']].sort_values('ranking_ano')
display(ra.style.format({'ranking_ano':'{:.0f}','DTP_RCL':'{:.2f}','Limite_Prudencial':'{:.2f}','Margem_Limite_Prudencial':'{:.2f}'}, na_rep='—').hide(axis='index').set_caption(f'Ranking anual — {ultimo_ano}'))
rh = indicadores[['ranking_historico','UF','nome_estado','score_fiscal','media_2015_2025','media_margem_prudencial','tendencia_recente','grupo']]
display(rh.style.format({'ranking_historico':'{:.0f}','score_fiscal':'{:.1f}','media_2015_2025':'{:.2f}','media_margem_prudencial':'{:.2f}'}, na_rep='—').hide(axis='index').set_caption('Ranking histórico multidimensional'))

## 4. Destaques e sinais de risco

As dimensões abaixo devem ser lidas separadamente. Um estado pode ter bom nível médio, mas estar em deterioração recente; outro pode ter indicador elevado, porém em trajetória de melhora.

**Melhora/piora** é a diferença entre o primeiro e o último exercício; **tendência recente** é a inclinação dos três últimos anos; **volatilidade** é o desvio-padrão da série; e **recorrência** conta os exercícios acima dos limites.

In [ ]:
quadros = [
 ('Cinco melhores scores','Maior equilíbrio no conjunto dos critérios.',indicadores.nsmallest(5,'ranking_historico'),['ranking_historico','UF','nome_estado','score_fiscal','grupo']),
 ('Cinco piores scores','Maior pressão relativa no conjunto dos critérios.',indicadores.nlargest(5,'ranking_historico'),['ranking_historico','UF','nome_estado','score_fiscal','grupo']),
 ('Maiores melhoras','Valores negativos indicam redução do DTP/RCL.',indicadores.nsmallest(5,'variacao_primeiro_ultimo_pp'),['UF','nome_estado','variacao_primeiro_ultimo_pp','media_2015_2025']),
 ('Maiores deteriorações','Valores positivos indicam aumento do DTP/RCL.',indicadores.nlargest(5,'variacao_primeiro_ultimo_pp'),['UF','nome_estado','variacao_primeiro_ultimo_pp','media_2015_2025']),
 ('Maior recorrência','Pressão persistente, mesmo quando o último ano melhora.',indicadores.sort_values(['anos_acima_maximo','anos_acima_prudencial','anos_acima_alerta'],ascending=False).head(5),['UF','nome_estado','anos_acima_alerta','anos_acima_prudencial','anos_acima_maximo']),
 ('Maior volatilidade','Oscilações elevadas reduzem a previsibilidade.',indicadores.nlargest(5,'volatilidade_desvio_padrao'),['UF','nome_estado','volatilidade_desvio_padrao','tendencia_recente']),
 ('Deterioração recente','Inclinação positiva indica aumento do DTP/RCL.',indicadores.nlargest(5,'tendencia_recente_pp_ano'),['UF','nome_estado','tendencia_recente_pp_ano','media_ultimos_3_anos']),
 ('Melhora recente','Inclinação negativa indica redução do DTP/RCL.',indicadores.nsmallest(5,'tendencia_recente_pp_ano'),['UF','nome_estado','tendencia_recente_pp_ano','media_ultimos_3_anos'])]
for titulo, texto, tabela, colunas in quadros:
    display(Markdown(f'### {titulo}\n{texto}'))
    formatos = {c:'{:.2f}' for c in colunas if c not in ['UF','nome_estado','grupo','tendencia_recente','ranking_historico','anos_acima_alerta','anos_acima_prudencial','anos_acima_maximo']}
    display(tabela[colunas].style.format(formatos, na_rep='—').hide(axis='index'))

## 5. Gráficos executivos

As visualizações são complementares: o ranking mostra a posição atual; as séries e o heatmap revelam persistência; as margens medem o espaço até os limites; e o score resume desempenho e risco históricos.

> Nos gráficos de margem, valores negativos indicam ultrapassagem. No ranking do score, valores maiores representam melhor situação relativa.

In [ ]:
generate_charts(base, indicadores)
comentarios = {
 '01_ranking_ultimo_ano':('Ranking no último ano','Barras menores indicam menor comprometimento; as cores refletem a situação fiscal.'),
 '02_evolucao_todos_estados':('Evolução de todos os estados','Revela mudanças de patamar, convergência e divergência ao longo do tempo.'),
 '03_evolucao_5_melhores_5_piores':('Trajetória dos extremos','Contrasta os cinco melhores e os cinco piores scores.'),
 '04_heatmap_estado_ano':('Mapa de calor Estado × Ano','Tons quentes indicam maior comprometimento e ajudam a reconhecer persistência.'),
 '05_distancia_limite_prudencial':('Margem prudencial','Mostra o espaço em pontos percentuais; valores negativos exigem atenção.'),
 '06_distancia_limite_maximo':('Margem máxima','Evidencia a distância do patamar legal mais crítico.'),
 '07_ranking_score_fiscal':('Ranking do score','Agrega nível, margens, recorrência, tendência e volatilidade.'),
 '08_comparacao_periodos':('Comparação entre períodos','Compara a média de 2015–2019 com 2020–2025.')}
for arquivo in sorted((ROOT / 'graficos').glob('*.png')):
    titulo, texto = comentarios.get(arquivo.stem, (arquivo.stem,''))
    display(Markdown(f'### {titulo}\n{texto}'))
    display(Image(filename=str(arquivo), width=950))

## 6. Limitações

- O escopo é o Poder Executivo e não a soma dos poderes;
- mudanças do leiaute podem afetar comparações, sobretudo antes e depois de 2018;
- população é contextual e não integra o score;
- o score é relativo às 27 UFs e muda quando a série é atualizada;
- eventos extraordinários devem ser investigados nos demonstrativos de cada ente;
- a análise apoia o monitoramento, mas não substitui avaliação jurídica, contábil ou de controle externo.

## 7. Síntese para a alta gestão

A síntese automática reúne cenário atual, diferenças regionais, extremos do score, trajetórias, recorrência e riscos que merecem acompanhamento.

In [ ]:
sintese = executive_summary(base, indicadores)
display(Markdown(sintese))
arquivo_excel = export_excel(base, indicadores, ranking)
display(Markdown(f'---\n**Excel atualizado:** `{arquivo_excel}`'))